# Week 4 — Day 6: Full Data Cleaning Pipeline

## Dataset: Melbourne Housing (Kaggle)
- Raw shape: 13,580 rows, 21 columns
- Missing cells: 4.6% overall
- Most missing: BuildingArea, YearBuilt, CouncilArea
- Final shape after cleaning: 6,496 rows, 22 columns

---

## Task 1 — ydata-profiling

Ran ProfileReport on raw data before writing any cleaning code.
Read full report to understand missingness, distributions, and warnings
before touching a single line of cleaning logic.

---

## Task 2 — src/cleaning.py Pipeline

Wrote cleaning logic as importable functions in src/cleaning.py
instead of notebook cells. This is a software engineering practice —
reusable, testable, and immediately signals quality to recruiters.

**Functions written:**

| Function | What it does |
|---|---|
| drop_high_missing() | Drops columns with >40% missing |
| impute_numeric() | Fills numeric nulls with median |
| impute_categorical() | Fills categorical nulls with mode |
| remove_iqr_outliers() | Removes rows outside Q1-1.5×IQR / Q3+1.5×IQR |
| parse_date() | Converts Date to Year + Month integers |
| clean() | Calls all functions in correct order |

**Bugs caught and fixed during development:**
- drop_high_missing: used () instead of [] for filtering — fixed to bracket indexing
- impute_numeric: select_dtypes was replacing entire df — fixed to use column names only
- impute_categorical: mode() returns DataFrame — fixed with .iloc[0]
- remove_iqr_outliers: must run BEFORE drop_high_missing or BuildingArea is already gone
- parse_date: Melbourne dates are DD/MM/YYYY — fixed with dayfirst=True

**Key insight:** Order of operations matters in a pipeline.
Dropping columns before removing outliers = KeyError on missing column.

---

## Task 3 — SQL Queries on Cleaned Data

Loaded df_clean into SQLite in-memory database. Wrote 5 queries:

### Q1 — Avg price by suburb (all 299 suburbs)
```sql
SELECT Suburb, AVG(Price) as avg_price
FROM melbourne
GROUP BY Suburb
ORDER BY avg_price DESC
```

### Q2 — Top 10 most expensive suburbs
```sql
SELECT Suburb, AVG(Price) as avg_price
FROM melbourne
GROUP BY Suburb
ORDER BY avg_price DESC
LIMIT 10
```
Result: Princes Hill ($1.9M) → Beaumaris ($1.56M)

### Q3 — Avg rooms per property type
```sql
SELECT Type, AVG(Rooms) as avg_rooms
FROM melbourne
GROUP BY Type
ORDER BY avg_rooms DESC
```
Result: Houses (3.2) > Townhouses (2.8) > Units (1.96)

### Q4 — Properties listed per month
```sql
SELECT Month, COUNT(*) as property_count
FROM melbourne
GROUP BY Month
ORDER BY Month ASC
```
Result: Peak in July (973), slowest in February (231)

### Q5 — Rank suburbs by avg price using window function
```sql
SELECT * FROM (
    SELECT Suburb, AVG(Price) as avg_price,
           ROW_NUMBER() OVER (ORDER BY AVG(Price) DESC) as rank
    FROM melbourne
    GROUP BY Suburb
)
```
Result: 299 suburbs ranked. Princes Hill #1, Kurunjang #299

---

## Task 4 — Baseline Random Forest Model

```python
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

# encode categoricals
cat_cols = df_model.select_dtypes(include='object').columns
for col in cat_cols:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

# 5-fold CV
rf = RandomForestRegressor(n_estimators=100, random_state=42)
scores = cross_val_score(rf, X, y, cv=5, scoring='neg_root_mean_squared_error')
```

**Result: CV RMSE = $195,259 ± $15,655**

---

## Data Story

### What was messy
- 13,580 rows, 21 columns with 4.6% missing cells
- BuildingArea and YearBuilt had highest missing %
- Date column in DD/MM/YYYY string format — needed parsing
- Price and BuildingArea had significant outliers

### What I did about it
- Removed IQR outliers from Price and BuildingArea first
- Dropped columns with >40% missing
- Median-imputed numeric, mode-imputed categorical columns
- Parsed Date into Year and Month integer features

### What surprised me
- Outlier removal cut dataset nearly in half (13,580 → 6,496 rows)
- Melbourne property market peaks in July — not summer months
- Princes Hill avg price ($1.9M) is 5x more than Kurunjang ($360k)
- Units average under 2 rooms — very small properties

### Next Steps (Week 5)
- Log-transform Price to fix right skew
- Proper suburb encoding (target encoding instead of label encoding)
- Add interaction features (Price per room, age of property)
- Expected RMSE improvement after feature engineering

---

## Key Takeaways
- Always profile before cleaning — understand before fixing
- Write cleaning logic in src/ not notebook cells — signals engineering mindset
- Pipeline order matters — outlier removal must come before column dropping
- dayfirst=True for DD/MM/YYYY date formats
- Baseline RMSE gives you a benchmark — every Week 5 change should beat $195,259
- Peak Melbourne listings in July is a real market insight, not just a SQL exercise

# Run ProfileReport

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# install ydata-profiling
import subprocess
subprocess.run(['pip', 'install', 'ydata-profiling', '--quiet'])

from ydata_profiling import ProfileReport

# load Melbourne Housing dataset
df = pd.read_csv('melb_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

# generate full profile report
profile = ProfileReport(df, title="Melbourne Housing EDA Report", 
                        explorative=True)
profile.to_file("melbourne_housing_profile.html")
print("Profile report saved to melbourne_housing_profile.html")

C:\Users\Kavya\AppData\Local\Temp\ipykernel_19172\302161067.py:10: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Dataset shape: (13580, 21)
Memory usage: 7493.11 KB


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 21/21 [00:00<00:00, 26.55it/s][A


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profile report saved to melbourne_housing_profile.html


# Create src/cleaning.py

In [2]:
import sys
sys.path.append('.')

from src.cleaning import clean
df_clean = clean(df.copy())
print(df_clean.shape)

(6496, 22)


# 5 SQL queries on the cleaned data

In [19]:
import sqlite3
conn = sqlite3.connect(':memory:')
df_clean.to_sql('melbourne', conn, index=False, if_exists='replace')
q1 = pd.read_sql("""
    SELECT Suburb, AVG(Price) as avg_price
    FROM melbourne
    GROUP BY Suburb
    ORDER BY avg_price DESC
""", conn)
print(q1)
q2 = pd.read_sql("""
    SELECT Suburb, AVG(Price) as avg_price
    FROM melbourne
    GROUP BY Suburb
    ORDER BY avg_price DESC
    LIMIT 10
""", conn)
print(q2)
q3 = pd.read_sql("""
    SELECT Type, AVG(Rooms) as avg_rooms
    FROM melbourne
    GROUP BY Type
    ORDER BY avg_rooms DESC
""", conn)
print(q3)
q4 = pd.read_sql("""
    SELECT Month, COUNT(*) as property_count
    FROM melbourne
    GROUP BY Month
    ORDER BY Month ASC
""", conn)
print(q4)
q5 = pd.read_sql("""
SELECT * FROM (
    SELECT Suburb, AVG(Price) as avg_price,
       ROW_NUMBER() OVER (ORDER BY AVG(Price) DESC) as rank
    FROM melbourne
    GROUP BY Suburb
)
""", conn)
print(q5)

           Suburb     avg_price
0    Princes Hill  1.904000e+06
1        Deepdene  1.680000e+06
2    Balwyn North  1.646615e+06
3     Albert Park  1.627907e+06
4     Middle Park  1.613077e+06
..            ...           ...
294     Truganina  4.205000e+05
295        Melton  3.956500e+05
296  Melton South  3.822778e+05
297        Wallan  3.660000e+05
298     Kurunjang  3.606667e+05

[299 rows x 2 columns]
         Suburb     avg_price
0  Princes Hill  1.904000e+06
1      Deepdene  1.680000e+06
2  Balwyn North  1.646615e+06
3   Albert Park  1.627907e+06
4   Middle Park  1.613077e+06
5     Ashburton  1.602983e+06
6   Sandringham  1.601464e+06
7     Parkville  1.572300e+06
8      Kew East  1.568059e+06
9     Beaumaris  1.558000e+06
  Type  avg_rooms
0    h   3.207108
1    t   2.826866
2    u   1.960051
    Month  property_count
0       2             231
1       3             282
2       4             480
3       5             941
4       6             820
5       7             973
6       

# Random Forest + Data Story.

In [20]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

# encode categorical columns first
df_model = df_clean.copy()
cat_cols = df_model.select_dtypes(include='object').columns
for col in cat_cols:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

# split features and target
X = df_model.drop(columns=['Price'])
y = df_model['Price']

# 5-fold CV
rf = RandomForestRegressor(n_estimators=100, random_state=42)
scores = cross_val_score(rf, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(f"CV RMSE: {-scores.mean():,.0f} ± {scores.std():,.0f}")

CV RMSE: 195,259 ± 15,655
